# Feature Engineering: PCA + Inclination Angle
This notebook demonstrates the key contribution of the project: engineering geometric features to improve classification accuracy from 0.82 to 0.92.

In [ ]:
import sys
sys.path.append('../src')
from supervised_classification import load_and_label_data, CLASS_NAMES
from feature_engineering import build_enhanced_features
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score
import matplotlib.pyplot as plt

sample_files = {
    1: '../data/sample_points/BuildingGableRoof_Train.las',
    2: '../data/sample_points/Vegetation_Train.las',
    3: '../data/sample_points/Ground_Train.las',
    4: '../data/sample_points/BuildingFlatRoof_Train.las',
}
points, labels, las = load_and_label_data('../data/sample_points/nonground_training.las', sample_files)

# Build enhanced features
features = build_enhanced_features(points, las.Z)

mask = labels[:, 0] != 0
X, y = features[mask], labels[mask, 0]
X_res, y_res = SMOTE(random_state=0).fit_resample(X, y)
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.3, random_state=0)

rf = RandomForestClassifier(n_estimators=800, max_depth=3, random_state=0)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f"Balanced Accuracy (Enhanced Features): {balanced_accuracy_score(y_test, y_pred):.4f}")

ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test, display_labels=CLASS_NAMES)
plt.title("Random Forest (PCA + Inclination Angle)")
plt.show()